# BF Development Factor Analysis — Emerging Market (v2)

This notebook documents how Bornhuetter-Ferguson (BF) development factors are derived and applied to the emerging market training data.

**Key findings:**
- Part 1 BF factors are **purely count-based (IBNR)** — derived from reported claim count triangles, NOT amount/severity development
- The emerging market has only 2 years (2023-2024) and a single incurred snapshot — cannot derive its own amount development factors
- The Part 1 factors from the developed market (5 years) are the best available proxy
- Open/closed analysis has **selection bias** (open claims are 1.3-2.8x more expensive than closed) — not a valid development signal

**This notebook does NOT modify any existing notebooks or submission files.**
It is a standalone analysis that produces:
1. Development triangles (count-based) for both developed and emerging markets
2. An Excel workbook with formulas documenting the BF factor derivation
3. Transparent documentation of all assumptions

## Methodology and Caveats

### What we CAN derive from the data:
1. **Reported claim COUNT triangle** (IBNR pattern) — using `loss_date` vs `report_date` to determine when claims were reported relative to the accident year. This is legitimate.
2. **Open/closed status snapshot** — shows current state of claims, but has **selection bias**.

### What we CANNOT derive:
1. **Amount development triangle** — we only have a single snapshot of `incurred_to_date` as-at the valuation date. No historical snapshots exist, so we cannot see how incurred amounts changed over time for the same claims.
2. **Severity development factors** — open/closed comparison is invalid because open claims are fundamentally different from closed claims (selection bias: small claims close quickly, large/complex claims stay open).

### How Part 1 factors were actually derived:
The Part 1 (developed market) BF factors are **count-based IBNR factors**, not amount development factors:
- For Motor BI: 2021 had 1,178 claims reported at lag 0, which eventually grew to 2,146 → factor = 2,146/1,178 = **1.82**
- For Subsidence: 2021 had 193 claims at lag 0, growing to 283 → factor = 283/193 = **1.47**
- For all other perils: similar count-based derivation
- These factors assume IBNR claims have the **same average severity** as already-reported claims
- There is **no severity development component** in the Part 1 factors

### Why Part 1 factors are used for emerging market:
- Emerging market has only 2 years (2023-2024) — not enough mature years for chain-ladder
- Single incurred snapshot — cannot build amount triangle
- Same insurance products, same perils → development patterns should be similar
- Emerging count link ratios are slightly higher than developed (less mature reporting) → Part 1 factors may even understate development needed

In [0]:
import pandas as pd
import numpy as np
import os

DATA_DEV = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/data/'
DATA_EM = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/data_emerging/'

# Developed market (5 years: 2020-2024)
dev = pd.read_parquet(DATA_DEV + 'claims.parquet')
dev['loss_date'] = pd.to_datetime(dev['loss_date'])
dev['report_date'] = pd.to_datetime(dev['report_date'])
dev['loss_year'] = dev['loss_date'].dt.year
dev['report_year'] = dev['report_date'].dt.year
dev['dev_lag'] = dev['report_year'] - dev['loss_year']
dev['incurred_net'] = dev['incurred_to_date'] - dev['recoveries']
dev = dev[dev['status'] != 'repudiated'].copy()
print(f"Developed market: {len(dev)} active claims, loss years {sorted(dev['loss_year'].unique())}")

# Emerging market (2 years: 2023-2024)
em = pd.read_parquet(DATA_EM + 'claims.parquet')
em['loss_date'] = pd.to_datetime(em['loss_date'])
em['report_date'] = pd.to_datetime(em['report_date'])
em['loss_year'] = em['loss_date'].dt.year
em['report_year'] = em['report_date'].dt.year
em['dev_lag'] = em['report_year'] - em['loss_year']
em['incurred_net'] = em['incurred_to_date'] - em['recoveries']
em = em[em['status'] != 'repudiated'].copy()
print(f"Emerging market: {len(em)} active claims, loss years {sorted(em['loss_year'].unique())}")

# Part 1 BF factors (from v4 notebook 01_Data_Prep_and_Claims, id 2899636018223423)
# These are COUNT-based IBNR factors, NOT amount development factors
# Derived from developed market 2021 mature year: ultimate_count / lag0_count
V1_FACTORS = {
    2021: {'bi': 1.00, 'tpp': 1.00, 'short_motor': 1.00, 'subsidence': 1.00, 'buildings_other': 1.00, 'contents': 1.00},
    2022: {'bi': 1.01, 'tpp': 1.01, 'short_motor': 1.01, 'subsidence': 1.05, 'buildings_other': 1.02, 'contents': 1.01},
    2023: {'bi': 1.06, 'tpp': 1.01, 'short_motor': 1.01, 'subsidence': 1.15, 'buildings_other': 1.02, 'contents': 1.01},
    2024: {'bi': 1.82, 'tpp': 1.08, 'short_motor': 1.08, 'subsidence': 1.47, 'buildings_other': 1.08, 'contents': 1.04},
}

def get_tail(peril, line):
    """Map peril to tail category for BF factor lookup."""
    p = str(peril).lower()
    if line == 'motor':
        if 'bodily' in p: return 'bi'
        elif 'third_party' in p: return 'tpp'
        else: return 'short_motor'
    elif line == 'buildings':
        if 'subsidence' in p: return 'subsidence'
        else: return 'buildings_other'
    else: return 'contents'

In [0]:
# Step 2: Build reported claim COUNT triangle — Developed Market
# This is the ONLY triangle we can legitimately build (using loss_date vs report_date)
# We CANNOT build an amount/severity triangle (single snapshot, no historical incurred)

print("="*70)
print("DEVELOPED MARKET — Reported Claim Count Triangle (Cumulative)")
print("="*70)

for line in ['motor', 'buildings', 'contents']:
    df = dev[dev['line'] == line]
    df = df[df['loss_year'] >= 2021]  # Skip sparse 2020
    count_inc = df.groupby(['loss_year', 'dev_lag']).size().unstack(fill_value=0)
    count_cum = count_inc.cumsum(axis=1)
    
    print(f"\n--- {line.upper()} ---")
    print("Incremental:")
    print(count_inc.to_string())
    print("Cumulative:")
    print(count_cum.to_string())
    
    # Link ratios
    print("Link ratios (lag n -> lag n+1):")
    for lag in range(count_cum.shape[1] - 1):
        col_curr = count_cum.columns[lag]
        col_next = count_cum.columns[lag + 1]
        vals = count_cum[[col_curr, col_next]].dropna()
        vals = vals[(vals[col_curr] > 0)]
        if len(vals) > 0:
            lr = vals[col_next].sum() / vals[col_curr].sum()
            print(f"  Lag {col_curr}->{col_next}: {lr:.4f} (based on {len(vals)} accident years)")

In [0]:
# Step 3: Per-peril count link ratios — Developed Market
# Shows IBNR by peril. Long-tail perils (BI, subsidence) have massive IBNR.

print("="*70)
print("DEVELOPED MARKET — Per-Peril Count Link Ratios (Lag 0->1)")
print("="*70)

peril_link = []
for (line, peril), grp in dev.groupby(['line', 'peril']):
    grp = grp[grp['loss_year'] >= 2021]
    count_inc = grp.groupby(['loss_year', 'dev_lag']).size().unstack(fill_value=0)
    count_cum = count_inc.cumsum(axis=1)
    if 0 not in count_cum.columns or 1 not in count_cum.columns:
        continue
    vals = count_cum[[0, 1]].dropna()
    vals = vals[(vals[0] > 0)]
    if len(vals) == 0:
        continue
    lr = vals[1].sum() / vals[0].sum()
    tail = get_tail(peril, line)
    peril_link.append({
        'line': line, 'peril': peril, 'tail': tail,
        'count_lag0': vals[0].sum(), 'count_lag1': vals[1].sum(),
        'link_ratio_0_1': lr, 'ibnr_pct': (lr - 1) * 100
    })

count_df = pd.DataFrame(peril_link)
print(count_df[['line', 'peril', 'tail', 'count_lag0', 'count_lag1', 'link_ratio_0_1', 'ibnr_pct']].to_string(index=False, float_format='%.4f'))
print("\nKey: BI has 51.7% IBNR, subsidence 31.6%, all others 1-5%")

In [0]:
# Step 4: VERIFY Part 1 Factor Derivation
# Part 1 Decision Log (notebook 04, id 2899636018223439) states:
#   "Bodily injury: 2,146/1,178 = 1.82 (54.9% at lag0)"
#   "Subsidence: 283/193 = 1.47 (68.2% at lag0)"
# These are COUNT-based: ultimate_claim_count / lag0_claim_count
# NOT amount-based. No severity development component.

print("="*70)
print("VERIFICATION: Part 1 BF Factor Derivation (COUNT-based IBNR)")
print("="*70)

for (line, peril, label) in [
    ('motor', 'bodily_injury', 'BI'),
    ('buildings', 'subsidence', 'Subsidence'),
    ('motor', 'third_party_prop', 'TPP'),
]:
    df = dev[(dev['line'] == line) & (dev['peril'] == peril)]
    df = df[df['loss_year'] >= 2021]
    count_inc = df.groupby(['loss_year', 'dev_lag']).size().unstack(fill_value=0)
    count_cum = count_inc.cumsum(axis=1)
    
    print(f"\n--- {label} ({line}/{peril}) ---")
    print(count_cum.to_string())
    
    # 2021: lag0 count vs ultimate count
    if 2021 in count_cum.index and 0 in count_cum.columns:
        lag0_2021 = count_cum.loc[2021, 0]
        ult_2021 = count_cum.loc[2021].iloc[-1]
        factor = ult_2021 / lag0_2021 if lag0_2021 > 0 else float('nan')
        pct_dev = 1 / factor if factor > 0 else float('nan')
        print(f"  2021: lag0={lag0_2021}, ultimate={ult_2021}, factor={factor:.4f} ({pct_dev*100:.1f}% developed at lag0)")
    
    # 2024: current lag0 count
    if 2024 in count_cum.index and 0 in count_cum.columns:
        lag0_2024 = count_cum.loc[2024, 0]
        print(f"  2024: lag0={lag0_2024} (apply factor to estimate ultimate = {lag0_2024 * factor:.0f})")

print("\n" + "="*70)
print("CONCLUSION: Part 1 factors are COUNT-based IBNR factors.")
print("They measure how many MORE claims get reported over time.")
print("They do NOT include severity development (open claims increasing in cost).")
print("They assume IBNR claims have the same avg severity as reported claims.")
print("="*70)

In [0]:
# Step 5: Build reported claim COUNT triangle — Emerging Market
# Only 2 years (2023-2024). Link ratio from 2023 row only.
# Emerging link ratios are slightly HIGHER than developed (less mature reporting).

print("="*70)
print("EMERGING MARKET — Reported Claim Count Triangle (Cumulative)")
print("="*70)

for line in ['motor', 'buildings', 'contents']:
    df = em[em['line'] == line]
    df = df[df['loss_year'] >= 2023]
    count_inc = df.groupby(['loss_year', 'dev_lag']).size().unstack(fill_value=0)
    count_cum = count_inc.cumsum(axis=1)
    
    print(f"\n--- {line.upper()} ---")
    print(count_cum.to_string())
    
    if 0 in count_cum.columns and 1 in count_cum.columns:
        vals = count_cum[[0, 1]].dropna()
        vals = vals[(vals[0] > 0)]
        if len(vals) > 0:
            lr = vals[1].sum() / vals[0].sum()
            print(f"  Link ratio (lag0->lag1): {lr:.4f} ({(lr-1)*100:.1f}% IBNR)")

# Per-peril for emerging
print("\n" + "="*70)
print("EMERGING MARKET — Per-Peril Count Link Ratios")
print("="*70)

emg_links = []
for (line, peril), grp in em.groupby(['line', 'peril']):
    grp = grp[grp['loss_year'] >= 2023]
    count_inc = grp.groupby(['loss_year', 'dev_lag']).size().unstack(fill_value=0)
    count_cum = count_inc.cumsum(axis=1)
    if 0 not in count_cum.columns or 1 not in count_cum.columns:
        continue
    vals = count_cum[[0, 1]].dropna()
    vals = vals[(vals[0] > 0)]
    if len(vals) == 0:
        continue
    lr = vals[1].sum() / vals[0].sum()
    tail = get_tail(peril, line)
    emg_links.append({'line': line, 'peril': peril, 'tail': tail,
                      'link_ratio': lr, 'ibnr_pct': (lr-1)*100})

emg_df = pd.DataFrame(emg_links)
print(emg_df[['line', 'peril', 'tail', 'link_ratio', 'ibnr_pct']].to_string(index=False, float_format='%.4f'))

In [0]:
# Step 6: Compare emerging vs developed count link ratios
# Emerging should have MORE IBNR (less mature market, slower reporting)

print("="*70)
print("COMPARISON: Emerging vs Developed Count Link Ratios (Lag 0->1)")
print("="*70)

# Developed per-peril
dev_links = {}
for (line, peril), grp in dev.groupby(['line', 'peril']):
    grp = grp[grp['loss_year'] >= 2021]
    ci = grp.groupby(['loss_year', 'dev_lag']).size().unstack(fill_value=0)
    cc = ci.cumsum(axis=1)
    if 0 in cc.columns and 1 in cc.columns:
        v = cc[[0,1]].dropna(); v = v[(v[0]>0)]
        if len(v) > 0:
            dev_links[(line, peril)] = v[1].sum() / v[0].sum()

# Emerging per-peril (already computed in emg_df)
emg_link_dict = {(r['line'], r['peril']): r['link_ratio'] for _, r in emg_df.iterrows()}

print(f"\n{'Line':12s} {'Peril':20s} {'Developed':>10s} {'Emerging':>10s} {'Diff':>10s} {'Notes'}")
print("-"*80)
for (line, peril), dl in sorted(dev_links.items()):
    el = emg_link_dict.get((line, peril), None)
    tail = get_tail(peril, line)
    if el:
        diff = el - dl
        note = 'Emerging has more IBNR' if diff > 0 else 'Emerging has less IBNR'
        if tail in ('bi', 'subsidence'): note += ' (long-tail)'
        print(f"{line:12s} {peril:20s} {dl:10.4f} {el:10.4f} {diff:+10.4f} {note}")
    else:
        print(f"{line:12s} {peril:20s} {dl:10.4f} {'N/A':>10s} {'':>10s}")

In [0]:
# Step 7: Open/Closed Analysis — WITH SELECTION BIAS WARNING
# CAUTION: Open claims are 1.3-2.8x MORE expensive than closed claims.
# This is selection bias (small claims close fast, large claims stay open),
# NOT a development signal. Do NOT use this to derive development factors.

print("="*70)
print("EMERGING MARKET — Open/Closed Snapshot (SELECTION BIAS WARNING)")
print("="*70)
print("WARNING: Open claims are bigger than closed — this is selection bias,")
print("not less development. Do NOT use open/closed to derive dev factors.")
print("="*70)

for line in ['motor', 'buildings', 'contents']:
    df = em[em['line'] == line]
    closed = df[df['status'] == 'closed']
    open_c = df[df['status'] == 'open']
    c_avg = closed['incurred_net'].mean()
    o_avg = open_c['incurred_net'].mean() if len(open_c) > 0 else 0
    ratio = o_avg / c_avg if c_avg > 0 else float('inf')
    print(f"\n{line.upper()}:")
    print(f"  Closed: n={len(closed):5d}, avg_incurred={c_avg:10,.2f}, paid/inc={closed['paid_to_date'].sum()/closed['incurred_to_date'].sum()*100:.1f}%")
    print(f"  Open:   n={len(open_c):5d}, avg_incurred={o_avg:10,.2f}, paid/inc={open_c['paid_to_date'].sum()/open_c['incurred_to_date'].sum()*100:.1f}%")
    print(f"  Open/Closed ratio: {ratio:.2f}x  <- selection bias, NOT development signal")

## BF Factor Documentation — Transparent Assumptions (v2_emerging)

### What the Part 1 factors actually are:

| Peril Group | 2024 Factor | Derivation | Type |
|---|---|---|---|
| Motor BI | 1.82 | 2,146/1,178 = count ultimate / count at lag0 | **Count-based IBNR** |
| Motor TPP | 1.08 | 8,193/7,573 = count ratio | **Count-based IBNR** |
| Motor short-tail | 1.08 | ~95% reported at lag0 | **Count-based IBNR** |
| Buildings subsidence | 1.47 | 283/193 = count ratio | **Count-based IBNR** |
| Buildings other | 1.08 | 93-98% reported at lag0 | **Count-based IBNR** |
| Contents all | 1.04 | 92-98% reported at lag0 | **Count-based IBNR** |

### Key assumption:
The Part 1 factors assume that **IBNR (unreported) claims have the same average severity as already-reported claims**. There is **no severity development component** — the factors only account for claims that haven't been reported yet.

### What this means:
- If IBNR claims are **more severe** than reported claims (e.g., BI claims that take years to report tend to be larger), the true development factor should be **higher** than 1.82
- If IBNR claims are **less severe**, the factor should be lower
- We cannot test this assumption with the available data (single snapshot)

### Why we use Part 1 factors for emerging market:
1. Emerging market has only 2 years — cannot derive its own factors
2. Same insurance products and perils → similar development patterns
3. Emerging count link ratios are slightly higher than developed → Part 1 factors may understate development
4. No better alternative available

In [0]:
# Step 8: Generate Excel Workbook with Formulas
# Creates BF_Development_Triangles.xlsx with 9 sheets showing:
# - Count triangles for both markets
# - Per-peril link ratios
# - Open/closed snapshot (with caveats)
# - BF factor application with Excel formulas

import subprocess
subprocess.check_call(['pip', 'install', 'openpyxl', '-q'])
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Styles
wb = Workbook()
HF = Font(name='Calibri', bold=True, size=11, color='FFFFFF')
HFILL = PatternFill(start_color='2F5496', end_color='2F5496', fill_type='solid')
TF = Font(name='Calibri', bold=True, size=14)
SF = Font(name='Calibri', bold=True, size=11)
NF = Font(name='Calibri', italic=True, size=10, color='666666')
WF = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')
TB = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))

def style_hdr(ws, row, cols):
    for c in range(1, cols + 1):
        cell = ws.cell(row=row, column=c)
        cell.font = HF; cell.fill = HFILL
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = TB

def auto_w(ws, min_w=10, max_w=28):
    for col in ws.columns:
        ml = 0; cl = get_column_letter(col[0].column)
        for cell in col:
            try:
                if cell.value and len(str(cell.value)) > ml: ml = len(str(cell.value))
            except: pass
        ws.column_dimensions[cl].width = min(max(ml + 2, min_w), max_w)

# === SHEET 1: README ===
ws = wb.active; ws.title = "README"
ws['A1'] = "Claims Development Triangle Analysis — Emerging Market (v2)"
ws['A1'].font = TF; ws.merge_cells('A1:H1')
readme_lines = [
    "", "PURPOSE",
    "Documents how BF development factors are derived and applied to emerging market.",
    "", "KEY FINDING",
    "Part 1 BF factors are COUNT-based IBNR factors (not amount/severity development).",
    "E.g., Motor BI factor 1.82 = 2,146 ultimate claims / 1,178 at lag 0.",
    "No severity development component — assumes IBNR claims have same avg severity as reported.",
    "", "WHAT WE CAN DERIVE FROM EMERGING DATA:",
    "  1. Reported claim COUNT triangle (IBNR) — from loss_date vs report_date",
    "  2. Open/closed snapshot — BUT has selection bias (open 1.3-2.8x more expensive)",
    "", "WHAT WE CANNOT DERIVE:",
    "  1. Amount development triangle (single snapshot, no historical incurred)",
    "  2. Severity development factors (selection bias invalidates open/closed)",
    "", "APPROACH:",
    "  1. Use developed market (5yr) count triangles to derive per-peril link ratios",
    "  2. Apply Part 1 BF factors to emerging market claims",
    "  3. Formula: ultimate = incurred_net x BF_factor",
    "", "SHEETS:",
    "  Summary, Dev_Count_Tri, Dev_Peril_Link, Dev_Severity,",
    "  Emg_Count_Tri, Emg_OpenClosed, Emg_vs_Dev, BF_Application",
]
for i, text in enumerate(readme_lines, start=2):
    ws.cell(row=i, column=1, value=text)
    if text and not text.startswith(' ') and text.endswith(':'):
        ws.cell(row=i, column=1).font = SF
ws.column_dimensions['A'].width = 85

print("README sheet done")

# === SHEET 2: Summary ===
ws2 = wb.create_sheet("Summary")
ws2['A1'] = "Part 1 BF Development Factors — Applied to Emerging Market"
ws2['A1'].font = TF; ws2.merge_cells('A1:G1')
ws2['A3'] = "COUNT-based IBNR factors from developed market (2020-2024)."
ws2['A3'].font = NF
ws2['A4'] = "Applied to emerging as: ultimate = incurred_net x dev_factor"
ws2['A4'].font = NF

hdrs = ['Line', 'Peril Group', 'Tail Type', '2023 Factor', '2024 Factor', '% Developed (2024)', 'Derivation']
for c, h in enumerate(hdrs, 1): ws2.cell(row=6, column=c, value=h)
style_hdr(ws2, 6, len(hdrs))

frows = [
    ('motor', 'bodily_injury', 'Long-tail (BI)', 1.06, 1.82, 0.549, '2146/1178 count ratio'),
    ('motor', 'third_party_prop', 'Medium-tail (TPP)', 1.01, 1.08, 0.926, '8193/7573 count ratio'),
    ('motor', 'glass_hail, own_damage, theft', 'Short-tail', 1.01, 1.08, 0.926, '~95% reported at lag0'),
    ('buildings', 'subsidence', 'Long-tail', 1.15, 1.47, 0.680, '283/193 count ratio'),
    ('buildings', 'fire, flood, storm, EoW, AD', 'Medium-tail', 1.02, 1.08, 0.926, '93-98% at lag0'),
    ('contents', 'all perils', 'Short-tail', 1.01, 1.04, 0.962, '92-98% at lag0'),
]
for r, row in enumerate(frows, start=7):
    for c, val in enumerate(row, 1):
        cell = ws2.cell(row=r, column=c, value=val); cell.border = TB
        if c in (4, 5): cell.number_format = '0.00'
        if c == 6: cell.number_format = '0.0%'

ws2['A15'] = "Impact on Emerging Market Ultimate Claims"
ws2['A15'].font = SF
ihdrs = ['Line', 'Reported Incurred (net)', 'Dev Uplift', 'Ultimate (v1)', 'Weighted Avg Factor']
for c, h in enumerate(ihdrs, 1): ws2.cell(row=17, column=c, value=h)
style_hdr(ws2, 17, len(ihdrs))

for r, line in enumerate(['motor', 'buildings', 'contents'], start=18):
    em_l = em[em['line'] == line].copy()
    em_l['tail'] = em_l.apply(lambda x: get_tail(x['peril'], x['line']), axis=1)
    em_l['vf'] = em_l.apply(lambda x: V1_FACTORS.get(x['loss_year'], {}).get(x['tail'], 1.0), axis=1)
    inc = em_l['incurred_net'].sum()
    wf = (em_l['incurred_net'] * em_l['vf']).sum() / inc if inc > 0 else 1.0
    ws2.cell(row=r, column=1, value=line).border = TB
    ws2.cell(row=r, column=2, value=inc).number_format = '#,##0'
    ws2.cell(row=r, column=3, value=f"=D{r}/B{r}-1").number_format = '0.0%'
    ws2.cell(row=r, column=4, value=f"=B{r}*(1+C{r})").number_format = '#,##0'
    ws2.cell(row=r, column=5, value=wf).number_format = '0.0000'
    for c in range(1, 6): ws2.cell(row=r, column=c).border = TB

ws2.cell(row=21, column=1, value='TOTAL').font = Font(bold=True)
ws2.cell(row=21, column=2, value="=SUM(B18:B20)").number_format = '#,##0'
ws2.cell(row=21, column=3, value="=D21/B21-1").number_format = '0.0%'
ws2.cell(row=21, column=4, value="=SUM(D18:D20)").number_format = '#,##0'
ws2.cell(row=21, column=5, value="=D21/B21").number_format = '0.0000'
for c in range(1, 6):
    ws2.cell(row=21, column=c).font = Font(bold=True)
    ws2.cell(row=21, column=c).border = TB
auto_w(ws2)
print("Summary sheet done")

# === SHEET 3: Dev_Count_Tri ===
ws3 = wb.create_sheet("Dev_Count_Tri")
ws3['A1'] = "Developed Market — Reported Claim Count Triangle (Cumulative)"
ws3['A1'].font = TF; ws3.merge_cells('A1:F1')
ws3['A2'] = "Source: developed market claims.parquet (2020-2024). Repudiated excluded."
ws3['A2'].font = NF

row_ptr = 4
for line in ['motor', 'buildings', 'contents']:
    df = dev[dev['line'] == line]
    df = df[df['loss_year'] >= 2021]
    count_inc = df.groupby(['loss_year', 'dev_lag']).size().unstack(fill_value=0)
    count_cum = count_inc.cumsum(axis=1)
    ws3.cell(row=row_ptr, column=1, value=f"{line.upper()} — Cumulative Reported Counts").font = SF
    row_ptr += 1
    ws3.cell(row=row_ptr, column=1, value="Accident Year")
    for c, lag in enumerate(count_cum.columns, 2): ws3.cell(row=row_ptr, column=c, value=f"Lag {lag}")
    ws3.cell(row=row_ptr, column=len(count_cum.columns) + 2, value="Link Ratio (0->1)")
    style_hdr(ws3, row_ptr, len(count_cum.columns) + 2)
    row_ptr += 1
    for ly in count_cum.index:
        ws3.cell(row=row_ptr, column=1, value=int(ly)).border = TB
        for c, lag in enumerate(count_cum.columns, 2):
            val = count_cum.loc[ly, lag]
            ws3.cell(row=row_ptr, column=c, value=int(val) if val > 0 else None).border = TB
        if 0 in count_cum.columns and 1 in count_cum.columns:
            b_col = get_column_letter(2); c_col = get_column_letter(3)
            if count_cum.loc[ly, 0] > 0:
                ws3.cell(row=row_ptr, column=len(count_cum.columns) + 2, value=f"={c_col}{row_ptr}/{b_col}{row_ptr}").number_format = '0.0000'
            ws3.cell(row=row_ptr, column=len(count_cum.columns) + 2).border = TB
        row_ptr += 1
    ws3.cell(row=row_ptr, column=1, value="Weighted Avg").font = Font(bold=True)
    if 0 in count_cum.columns and 1 in count_cum.columns:
        ds = row_ptr - len(count_cum.index); de = row_ptr - 1
        b_col = get_column_letter(2); c_col = get_column_letter(3)
        ws3.cell(row=row_ptr, column=len(count_cum.columns) + 2, value=f"=SUM({c_col}{ds}:{c_col}{de})/SUM({b_col}{ds}:{b_col}{de})").number_format = '0.0000'
    for c in range(1, len(count_cum.columns) + 3):
        ws3.cell(row=row_ptr, column=c).border = TB; ws3.cell(row=row_ptr, column=c).font = Font(bold=True)
    row_ptr += 2
auto_w(ws3)
print("Dev_Count_Tri sheet done")

# === SHEET 4: Dev_Peril_Link ===
ws4 = wb.create_sheet("Dev_Peril_Link")
ws4['A1'] = "Developed Market — Per-Peril Count Link Ratios (Lag 0->1)"
ws4['A1'].font = TF; ws4.merge_cells('A1:G1')
ws4['A2'] = "IBNR = Incurred But Not Reported. Link ratio > 1 means some claims reported late."
ws4['A2'].font = NF
hdrs = ['Line', 'Peril', 'Count at Lag 0', 'Count at Lag 1', 'Link Ratio', 'IBNR %', 'Tail Type']
for c, h in enumerate(hdrs, 1): ws4.cell(row=4, column=c, value=h)
style_hdr(ws4, 4, len(hdrs))
row_ptr = 5
tail_labels = {'bi': 'Long-tail (BI)', 'tpp': 'Medium-tail (TPP)', 'short_motor': 'Short-tail', 'subsidence': 'Long-tail', 'buildings_other': 'Medium-tail', 'contents': 'Short-tail'}
for (line, peril), grp in dev.groupby(['line', 'peril']):
    grp = grp[grp['loss_year'] >= 2021]
    ci = grp.groupby(['loss_year', 'dev_lag']).size().unstack(fill_value=0)
    cc = ci.cumsum(axis=1)
    if 0 not in cc.columns or 1 not in cc.columns: continue
    v = cc[[0,1]].dropna(); v = v[(v[0]>0)]
    if len(v) == 0: continue
    c0 = v[0].sum(); c1 = v[1].sum()
    tail = get_tail(peril, line)
    ws4.cell(row=row_ptr, column=1, value=line).border = TB
    ws4.cell(row=row_ptr, column=2, value=peril).border = TB
    ws4.cell(row=row_ptr, column=3, value=int(c0)).border = TB
    ws4.cell(row=row_ptr, column=4, value=int(c1)).border = TB
    ws4.cell(row=row_ptr, column=5, value=f"=D{row_ptr}/C{row_ptr}").number_format = '0.0000'
    ws4.cell(row=row_ptr, column=5).border = TB
    ws4.cell(row=row_ptr, column=6, value=f"=(E{row_ptr}-1)").number_format = '0.00%'
    ws4.cell(row=row_ptr, column=6).border = TB
    ws4.cell(row=row_ptr, column=7, value=tail_labels.get(tail, '')).border = TB
    if tail in ('bi', 'subsidence'):
        for c in range(1, 8): ws4.cell(row=row_ptr, column=c).fill = WF
    row_ptr += 1
auto_w(ws4)
print("Dev_Peril_Link sheet done")

# === SHEET 5: Dev_Severity ===
ws5 = wb.create_sheet("Dev_Severity")
ws5['A1'] = "Developed Market — Closed Claim Severity by Loss Year"
ws5['A1'].font = TF; ws5.merge_cells('A1:I1')
ws5['A2'] = "Shows severity by year and % closed. CAUTION: open claims are MORE expensive (selection bias)."
ws5['A2'].font = NF
row_ptr = 4
for line in ['motor', 'buildings', 'contents']:
    ws5.cell(row=row_ptr, column=1, value=f"{line.upper()}").font = SF; row_ptr += 1
    hdrs = ['Peril', 'Loss Year', 'Total', 'Closed', 'Open', '% Closed', 'Closed Avg', 'Open Avg', 'Open/Closed']
    for c, h in enumerate(hdrs, 1): ws5.cell(row=row_ptr, column=c, value=h)
    style_hdr(ws5, row_ptr, len(hdrs)); row_ptr += 1
    line_df = dev[dev['line'] == line]
    for peril in sorted(line_df['peril'].unique()):
        peril_df = line_df[line_df['peril'] == peril]
        for yr in sorted(peril_df['loss_year'].unique()):
            yr_df = peril_df[peril_df['loss_year'] == yr]
            closed = yr_df[yr_df['status'] == 'closed']
            open_c = yr_df[yr_df['status'] == 'open']
            ws5.cell(row=row_ptr, column=1, value=peril).border = TB
            ws5.cell(row=row_ptr, column=2, value=int(yr)).border = TB
            ws5.cell(row=row_ptr, column=3, value=len(yr_df)).border = TB
            ws5.cell(row=row_ptr, column=4, value=len(closed)).border = TB
            ws5.cell(row=row_ptr, column=5, value=len(open_c)).border = TB
            ws5.cell(row=row_ptr, column=6, value=f"=D{row_ptr}/C{row_ptr}").number_format = '0.0%'
            ws5.cell(row=row_ptr, column=6).border = TB
            c_avg = closed['incurred_net'].mean() if len(closed) > 0 else 0
            o_avg = open_c['incurred_net'].mean() if len(open_c) > 0 else 0
            ws5.cell(row=row_ptr, column=7, value=round(c_avg, 2)).number_format = '#,##0'
            ws5.cell(row=row_ptr, column=7).border = TB
            ws5.cell(row=row_ptr, column=8, value=round(o_avg, 2) if o_avg > 0 else None).number_format = '#,##0'
            ws5.cell(row=row_ptr, column=8).border = TB
            if c_avg > 0 and o_avg > 0:
                ws5.cell(row=row_ptr, column=9, value=f"=H{row_ptr}/G{row_ptr}").number_format = '0.00"x"'
            ws5.cell(row=row_ptr, column=9).border = TB
            row_ptr += 1
    row_ptr += 1
auto_w(ws5)
print("Dev_Severity sheet done")

In [0]:
# === SHEET 6: Emg_Count_Tri ===
ws6 = wb.create_sheet("Emg_Count_Tri")
ws6['A1'] = "Emerging Market — Reported Claim Count Triangle (Cumulative)"
ws6['A1'].font = TF; ws6.merge_cells('A1:F1')
ws6['A2'] = "Only 2 years (2023-2024). Link ratios from 2023 row only."
ws6['A2'].font = NF
row_ptr = 4
for line in ['motor', 'buildings', 'contents']:
    df = em[em['line'] == line]
    df = df[df['loss_year'] >= 2023]
    count_inc = df.groupby(['loss_year', 'dev_lag']).size().unstack(fill_value=0)
    count_cum = count_inc.cumsum(axis=1)
    ws6.cell(row=row_ptr, column=1, value=f"{line.upper()} — Cumulative Reported Counts").font = SF; row_ptr += 1
    ws6.cell(row=row_ptr, column=1, value="Accident Year")
    for c, lag in enumerate(count_cum.columns, 2): ws6.cell(row=row_ptr, column=c, value=f"Lag {lag}")
    ws6.cell(row=row_ptr, column=len(count_cum.columns) + 2, value="Link Ratio (0->1)")
    style_hdr(ws6, row_ptr, len(count_cum.columns) + 2); row_ptr += 1
    for ly in count_cum.index:
        ws6.cell(row=row_ptr, column=1, value=int(ly)).border = TB
        for c, lag in enumerate(count_cum.columns, 2):
            val = count_cum.loc[ly, lag]
            ws6.cell(row=row_ptr, column=c, value=int(val) if val > 0 else None).border = TB
        if 0 in count_cum.columns and 1 in count_cum.columns:
            b_col = get_column_letter(2); c_col = get_column_letter(3)
            if count_cum.loc[ly, 0] > 0:
                ws6.cell(row=row_ptr, column=len(count_cum.columns) + 2, value=f"={c_col}{row_ptr}/{b_col}{row_ptr}").number_format = '0.0000'
            ws6.cell(row=row_ptr, column=len(count_cum.columns) + 2).border = TB
        row_ptr += 1
    ws6.cell(row=row_ptr, column=1, value="Weighted Avg").font = Font(bold=True)
    if 0 in count_cum.columns and 1 in count_cum.columns:
        ds = row_ptr - len(count_cum.index); de = row_ptr - 1
        b_col = get_column_letter(2); c_col = get_column_letter(3)
        ws6.cell(row=row_ptr, column=len(count_cum.columns) + 2, value=f"=SUM({c_col}{ds}:{c_col}{de})/SUM({b_col}{ds}:{b_col}{de})").number_format = '0.0000'
    for c in range(1, len(count_cum.columns) + 3):
        ws6.cell(row=row_ptr, column=c).border = TB; ws6.cell(row=row_ptr, column=c).font = Font(bold=True)
    row_ptr += 2

# Per-peril for emerging
ws6.cell(row=row_ptr, column=1, value="Per-Peril Count Link Ratios (Emerging)").font = SF; row_ptr += 1
hdrs = ['Line', 'Peril', 'Count Lag 0', 'Count Lag 1', 'Link Ratio', 'IBNR %']
for c, h in enumerate(hdrs, 1): ws6.cell(row=row_ptr, column=c, value=h)
style_hdr(ws6, row_ptr, len(hdrs)); row_ptr += 1
for (line, peril), grp in em.groupby(['line', 'peril']):
    grp = grp[grp['loss_year'] >= 2023]
    ci = grp.groupby(['loss_year', 'dev_lag']).size().unstack(fill_value=0)
    cc = ci.cumsum(axis=1)
    if 0 not in cc.columns or 1 not in cc.columns: continue
    v = cc[[0,1]].dropna(); v = v[(v[0]>0)]
    if len(v) == 0: continue
    c0 = v[0].sum(); c1 = v[1].sum()
    ws6.cell(row=row_ptr, column=1, value=line).border = TB
    ws6.cell(row=row_ptr, column=2, value=peril).border = TB
    ws6.cell(row=row_ptr, column=3, value=int(c0)).border = TB
    ws6.cell(row=row_ptr, column=4, value=int(c1)).border = TB
    ws6.cell(row=row_ptr, column=5, value=f"=D{row_ptr}/C{row_ptr}").number_format = '0.0000'
    ws6.cell(row=row_ptr, column=5).border = TB
    ws6.cell(row=row_ptr, column=6, value=f"=(E{row_ptr}-1)").number_format = '0.00%'
    ws6.cell(row=row_ptr, column=6).border = TB
    tail = get_tail(peril, line)
    if tail in ('bi', 'subsidence'):
        for c in range(1, 7): ws6.cell(row=row_ptr, column=c).fill = WF
    row_ptr += 1
auto_w(ws6)
print("Emg_Count_Tri sheet done")

# === SHEET 7: Emg_OpenClosed ===
ws7 = wb.create_sheet("Emg_OpenClosed")
ws7['A1'] = "Emerging Market — Open/Closed Snapshot (CAVEAT: Selection Bias)"
ws7['A1'].font = TF; ws7.merge_cells('A1:I1')
ws7['A2'] = "WARNING: Open claims are 1.3-2.8x MORE expensive than closed. Selection bias, NOT development."
ws7['A2'].font = Font(name='Calibri', italic=True, size=10, color='CC0000')
ws7['A3'] = "Do NOT use open/closed averages to derive development factors."
ws7['A3'].font = Font(name='Calibri', italic=True, size=10, color='CC0000')
row_ptr = 5
for line in ['motor', 'buildings', 'contents']:
    ws7.cell(row=row_ptr, column=1, value=f"{line.upper()}").font = SF; row_ptr += 1
    hdrs = ['Peril', 'Loss Year', 'Total', 'Closed', 'Open', '% Closed', 'Closed Avg', 'Open Avg', 'Open/Closed']
    for c, h in enumerate(hdrs, 1): ws7.cell(row=row_ptr, column=c, value=h)
    style_hdr(ws7, row_ptr, len(hdrs)); row_ptr += 1
    line_df = em[em['line'] == line]
    for peril in sorted(line_df['peril'].unique()):
        peril_df = line_df[line_df['peril'] == peril]
        for yr in sorted(peril_df['loss_year'].unique()):
            if yr < 2023: continue
            yr_df = peril_df[peril_df['loss_year'] == yr]
            closed = yr_df[yr_df['status'] == 'closed']
            open_c = yr_df[yr_df['status'] == 'open']
            ws7.cell(row=row_ptr, column=1, value=peril).border = TB
            ws7.cell(row=row_ptr, column=2, value=int(yr)).border = TB
            ws7.cell(row=row_ptr, column=3, value=len(yr_df)).border = TB
            ws7.cell(row=row_ptr, column=4, value=len(closed)).border = TB
            ws7.cell(row=row_ptr, column=5, value=len(open_c)).border = TB
            ws7.cell(row=row_ptr, column=6, value=f"=D{row_ptr}/C{row_ptr}").number_format = '0.0%'
            ws7.cell(row=row_ptr, column=6).border = TB
            c_avg = closed['incurred_net'].mean() if len(closed) > 0 else 0
            o_avg = open_c['incurred_net'].mean() if len(open_c) > 0 else 0
            ws7.cell(row=row_ptr, column=7, value=round(c_avg, 2)).number_format = '#,##0'
            ws7.cell(row=row_ptr, column=7).border = TB
            ws7.cell(row=row_ptr, column=8, value=round(o_avg, 2) if o_avg > 0 else None).number_format = '#,##0'
            ws7.cell(row=row_ptr, column=8).border = TB
            if c_avg > 0 and o_avg > 0:
                ws7.cell(row=row_ptr, column=9, value=f"=H{row_ptr}/G{row_ptr}").number_format = '0.00"x"'
            ws7.cell(row=row_ptr, column=9).border = TB
            row_ptr += 1
    row_ptr += 1
auto_w(ws7)
print("Emg_OpenClosed sheet done")

# === SHEET 8: Emg_vs_Dev ===
ws8 = wb.create_sheet("Emg_vs_Dev")
ws8['A1'] = "Comparison: Emerging vs Developed Count Link Ratios (Lag 0->1)"
ws8['A1'].font = TF; ws8.merge_cells('A1:F1')
ws8['A2'] = "Emerging link ratios slightly HIGHER (more IBNR) — less mature reporting."
ws8['A2'].font = NF
hdrs = ['Line', 'Peril', 'Developed Link', 'Emerging Link', 'Difference', 'Notes']
for c, h in enumerate(hdrs, 1): ws8.cell(row=4, column=c, value=h)
style_hdr(ws8, 4, len(hdrs))
dev_links = {}
for (line, peril), grp in dev.groupby(['line', 'peril']):
    grp = grp[grp['loss_year'] >= 2021]
    ci = grp.groupby(['loss_year', 'dev_lag']).size().unstack(fill_value=0)
    cc = ci.cumsum(axis=1)
    if 0 in cc.columns and 1 in cc.columns:
        v = cc[[0,1]].dropna(); v = v[(v[0]>0)]
        if len(v) > 0: dev_links[(line, peril)] = v[1].sum() / v[0].sum()
emg_links = {(r['line'], r['peril']): r['link_ratio'] for _, r in emg_df.iterrows()}
row_ptr = 5
for (line, peril) in sorted(set(list(dev_links.keys()) + list(emg_links.keys())), key=lambda x: (x[0], x[1])):
    dl = dev_links.get((line, peril), None)
    el = emg_links.get((line, peril), None)
    tail = get_tail(peril, line)
    notes = ''
    if tail == 'bi': notes = 'Long-tail: massive IBNR in both'
    elif tail == 'subsidence': notes = 'Long-tail: high IBNR in both'
    elif dl and el and el > dl: notes = 'Emerging has more IBNR (less mature)'
    ws8.cell(row=row_ptr, column=1, value=line).border = TB
    ws8.cell(row=row_ptr, column=2, value=peril).border = TB
    ws8.cell(row=row_ptr, column=3, value=round(dl, 4) if dl else None).number_format = '0.0000'
    ws8.cell(row=row_ptr, column=3).border = TB
    ws8.cell(row=row_ptr, column=4, value=round(el, 4) if el else None).number_format = '0.0000'
    ws8.cell(row=row_ptr, column=4).border = TB
    if dl and el:
        ws8.cell(row=row_ptr, column=5, value=f"=D{row_ptr}-C{row_ptr}").number_format = '+0.0000;-0.0000'
    ws8.cell(row=row_ptr, column=5).border = TB
    ws8.cell(row=row_ptr, column=6, value=notes).border = TB
    if tail in ('bi', 'subsidence'):
        for c in range(1, 7): ws8.cell(row=row_ptr, column=c).fill = WF
    row_ptr += 1
auto_w(ws8)
print("Emg_vs_Dev sheet done")

# === SHEET 9: BF_Application ===
ws9 = wb.create_sheet("BF_Application")
ws9['A1'] = "BF Factor Application to Emerging Claims — By Peril and Loss Year"
ws9['A1'].font = TF; ws9.merge_cells('A1:K1')
ws9['A2'] = "Formula: ultimate = incurred_net x BF_factor. BF factors are COUNT-based IBNR from developed market."
ws9['A2'].font = NF
row_ptr = 5
tail_labels = {'bi': 'Long-tail (BI)', 'tpp': 'Medium-tail', 'short_motor': 'Short-tail', 'subsidence': 'Long-tail', 'buildings_other': 'Medium-tail', 'contents': 'Short-tail'}
for line in ['motor', 'buildings', 'contents']:
    ws9.cell(row=row_ptr, column=1, value=f"{line.upper()}").font = SF; row_ptr += 1
    hdrs = ['Peril', 'Tail Type', 'Loss Year', 'Claims', 'Incurred (net)', 'BF Factor', 'Ultimate', 'IBNR Amount', '% Developed', 'Derivation']
    for c, h in enumerate(hdrs, 1): ws9.cell(row=row_ptr, column=c, value=h)
    style_hdr(ws9, row_ptr, len(hdrs)); row_ptr += 1
    line_df = em[em['line'] == line]
    for peril in sorted(line_df['peril'].unique()):
        peril_df = line_df[line_df['peril'] == peril]
        tail = get_tail(peril, line)
        for yr in [2023, 2024]:
            yr_df = peril_df[peril_df['loss_year'] == yr]
            if len(yr_df) == 0: continue
            inc = yr_df['incurred_net'].sum()
            factor = V1_FACTORS.get(yr, {}).get(tail, 1.0)
            deriv = {'bi': f'count ratio (2146/1178)' if yr == 2024 else 'partial dev', 'subsidence': f'count ratio (283/193)' if yr == 2024 else 'partial dev'}.get(tail, f'{1/factor*100:.0f}% reported')
            ws9.cell(row=row_ptr, column=1, value=peril).border = TB
            ws9.cell(row=row_ptr, column=2, value=tail_labels.get(tail, '')).border = TB
            ws9.cell(row=row_ptr, column=3, value=yr).border = TB
            ws9.cell(row=row_ptr, column=4, value=len(yr_df)).border = TB
            ws9.cell(row=row_ptr, column=5, value=round(inc, 2)).number_format = '#,##0'
            ws9.cell(row=row_ptr, column=5).border = TB
            ws9.cell(row=row_ptr, column=6, value=factor).number_format = '0.00'
            ws9.cell(row=row_ptr, column=6).border = TB
            ws9.cell(row=row_ptr, column=7, value=f"=E{row_ptr}*F{row_ptr}").number_format = '#,##0'
            ws9.cell(row=row_ptr, column=7).border = TB
            ws9.cell(row=row_ptr, column=8, value=f"=G{row_ptr}-E{row_ptr}").number_format = '#,##0'
            ws9.cell(row=row_ptr, column=8).border = TB
            ws9.cell(row=row_ptr, column=9, value=f"=1/F{row_ptr}").number_format = '0.0%'
            ws9.cell(row=row_ptr, column=9).border = TB
            ws9.cell(row=row_ptr, column=10, value=deriv).border = TB
            if tail in ('bi', 'subsidence') and yr == 2024:
                for c in range(1, 11): ws9.cell(row=row_ptr, column=c).fill = WF
            row_ptr += 1
    row_ptr += 1
auto_w(ws9)
print("BF_Application sheet done")

# Save
OUTPUT_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/Excel files_emerging_v2/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, 'BF_Development_Triangles.xlsx')
wb.save(output_path)
print(f"\nWorkbook saved: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"Sheets: {wb.sheetnames}")